Medallion Architecture

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.ecommerce.bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.ecommerce.silver")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.ecommerce.gold")

DataFrame[]

In [0]:
spark.sql("SHOW VOLUMES IN workspace.ecommerce").show()


+---------+--------------+
| database|   volume_name|
+---------+--------------+
|ecommerce|        bronze|
|ecommerce|         delta|
|ecommerce|ecommerce_data|
|ecommerce|          gold|
|ecommerce|        silver|
+---------+--------------+



In [0]:
bronze_path = "/Volumes/workspace/ecommerce/bronze/events"
silver_path = "/Volumes/workspace/ecommerce/silver/events"
gold_path   = "/Volumes/workspace/ecommerce/gold/product_performance"


In [0]:
bronze.write.format("delta").mode("overwrite").save(bronze_path)


In [0]:
from pyspark.sql import functions as F

bronze_path = "/Volumes/workspace/ecommerce/bronze/events"
silver_path = "/Volumes/workspace/ecommerce/silver/events"

# Load Bronze
bronze = spark.read.format("delta").load(bronze_path)

# Create Silver
silver = (
    bronze
    .filter(F.col("price").isNotNull())
    .filter(F.col("price") > 0)
    .filter(F.col("price") < 10000)
    .dropDuplicates(["user_session", "event_time"])
    .withColumn("event_date", F.to_date("event_time"))
    .withColumn(
        "price_tier",
        F.when(F.col("price") < 10, "budget")
         .when(F.col("price") < 50, "mid")
         .otherwise("premium")
    )
)

# Write Silver
silver.write.format("delta") \
    .mode("overwrite") \
    .save(silver_path)

print("SILVER layer created successfully")


SILVER layer created successfully


In [0]:
gold_path = "/Volumes/workspace/ecommerce/gold/product_performance"

# Create Gold aggregates
gold = (
    silver
    .groupBy("product_id")
    .agg(
        F.count(F.when(F.col("event_type") == "view", True)).alias("views"),
        F.count(F.when(F.col("event_type") == "purchase", True)).alias("purchases"),
        F.round(
            F.sum(F.when(F.col("event_type") == "purchase", F.col("price"))), 2
        ).alias("revenue")
    )
    .withColumn(
        "conversion_rate",
        F.expr("try_divide(purchases, views) * 100")
    )
)

gold.write.format("delta") \
    .mode("overwrite") \
    .save(gold_path)

print("GOLD layer created successfully")

GOLD layer created successfully


In [0]:
silver.count()
gold.count()


165646